In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, LongType, ArrayType
)
from pyspark.sql.functions import current_timestamp, input_file_name, col

In [0]:
# ---- schema ----
incident_update_schema = StructType([
    StructField("id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("body", StringType(), True),
    StructField("created_at", StringType(), True),
])

incident_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("impact", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("incident_updates", ArrayType(incident_update_schema), True),
])

component_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("group", BooleanType(), True),
    StructField("group_id", StringType(), True),
    StructField("position", LongType(), True),
    StructField("description", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
])

page_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("url", StringType(), True),
    StructField("time_zone", StringType(), True),
    StructField("updated_at", StringType(), True),
])

status_schema = StructType([
    StructField("indicator", StringType(), True),
    StructField("description", StringType(), True),
])

status_page_schema = StructType([
    StructField("_ingested_at", StringType(), True),
    StructField("page", page_schema, True),
    StructField("status", status_schema, True),
    StructField("components", ArrayType(component_schema), True),
    StructField("incidents", ArrayType(incident_schema), True),
    StructField("scheduled_maintenances", ArrayType(incident_schema), True),
])

In [0]:
# ---- paths ----
raw_path = "s3://vendor-status-monitor/raw/status_checks/"
checkpoint_path = "s3://vendor-status-monitor/checkpoints/bronze/status_checks/"
bronze_table = "vendor_status.bronze.status_checks"

# ---- stream read + write ----
bronze_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(status_page_schema)
    .load(raw_path)
    .withColumn("_bronze_loaded_at", current_timestamp())
    #.withColumn("_source_file", input_file_name())
    .withColumn("_source_file", col("_metadata.file_path"))
)

(
    bronze_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)